# 注意力汇聚：Nadaraya-Watson 核回归

本笔记本实现了带参数的注意力汇聚模型，来自 [d2l.ai](http://zh-v2.d2l.ai/chapter_attention-mechanisms/nadaraya-waston.html) 第10.2节。

主要内容：
- 生成数据集
- 非参数注意力汇聚（作为对比）
- 带参数注意力汇聚
- 训练与可视化


## 1. 导入必要的库


In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子以保证可重复性
torch.manual_seed(42)


## 2. 生成数据集

使用以下非线性函数生成一个人工数据集：
$$y_i = 2\sin(x_i) + x_i^{0.8} + \epsilon$$

其中 $\epsilon$ 是服从均值为0、标准差为0.5的正态分布的噪声。


In [ ]:
n_train = 50  # 训练样本数
x_train, _ = torch.sort(torch.rand(n_train) * 5)  # 排序后的训练样本

def f(x):
    """真实的非线性函数"""
    return 2 * torch.sin(x) + x ** 0.8

y_train = f(x_train) + torch.normal(0.0, 0.5, (n_train,))  # 训练样本的输出（带噪声）
x_test = torch.arange(0, 5, 0.1)  # 测试样本
y_truth = f(x_test)  # 测试样本的真实输出
n_test = len(x_test)  # 测试样本数

print(f'训练样本数: {n_train}')
print(f'测试样本数: {n_test}')


## 3. 绘图辅助函数


In [ ]:
def plot_kernel_reg(y_hat):
    """绘制核回归结果"""
    plt.figure(figsize=(10, 6))
    plt.plot(x_test.numpy(), y_truth.numpy(), label='Truth', linewidth=2)
    plt.plot(x_test.numpy(), y_hat.detach().numpy(), '--', label='Pred', linewidth=2)
    plt.scatter(x_train.numpy(), y_train.numpy(), c='red', s=30, label='Train', alpha=0.7)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title('Nadaraya-Watson 核回归')
    plt.show()

def show_heatmaps(matrices, xlabel='', ylabel='', titles=None, figsize=(6, 6), cmap='Reds'):
    """显示注意力权重热力图"""
    num_rows, num_cols = matrices.shape[0], matrices.shape[1]
    fig, axes = plt.subplots(num_rows, num_cols, figsize=figsize, squeeze=False)
    for i in range(num_rows):
        for j in range(num_cols):
            ax = axes[i][j]
            pcm = ax.imshow(matrices[i, j].detach().numpy(), cmap=cmap)
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            if titles:
                ax.set_title(titles[j])
    fig.colorbar(pcm, ax=axes, shrink=0.6)
    plt.tight_layout()
    plt.show()


## 4. 非参数注意力汇聚（作为对比）

基于 Nadaraya-Watson 核回归的非参数模型：
$$f(x) = \sum_{i=1}^{n} \frac{K(x - x_i)}{\sum_{j=1}^{n} K(x - x_j)} y_i$$

其中 $K$ 是高斯核。


In [ ]:
# 使用高斯核的非参数注意力汇聚
# X_repeat的形状:(n_test, n_train), 每一行包含相同的测试输入
X_repeat = x_test.repeat_interleave(n_train).reshape((-1, n_train))
# x_train的形状:(n_train,)。相减后得到形状为(n_test, n_train)的矩阵
# 其中每一行表示每个测试输入与所有训练输入的差
attention_weights = nn.functional.softmax(-(X_repeat - x_train) ** 2 / 2, dim=1)
# y_hat的形状:(n_test,)
y_hat = torch.matmul(attention_weights, y_train)

plot_kernel_reg(y_hat)


In [ ]:
# 可视化非参数模型的注意力权重
show_heatmaps(attention_weights.unsqueeze(0).unsqueeze(0),
              xlabel='Sorted training inputs',
              ylabel='Sorted testing inputs')


## 5. 批量矩阵乘法

为了更高效地计算小批量数据的注意力，我们使用批量矩阵乘法 `torch.bmm`。


In [ ]:
# 批量矩阵乘法示例
X = torch.ones((2, 1, 4))
Y = torch.ones((2, 4, 6))
result = torch.bmm(X, Y)
print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')
print(f'bmm(X, Y) shape: {result.shape}')


## 6. 带参数注意力汇聚模型

在注意力框架下，使用可学习的参数 $w$ 来改进 Nadaraya-Watson 核回归：

$$f(x) = \sum_{i=1}^{n} \alpha(x, x_i) y_i = \sum_{i=1}^{n} \frac{\exp\left(-\frac{1}{2}((x - x_i)w)^2\right)}{\sum_{j=1}^{n} \exp\left(-\frac{1}{2}((x - x_j)w)^2\right)} y_i$$


In [ ]:
class NWKernelRegression(nn.Module):
    """带参数的 Nadaraya-Watson 核回归模型"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.w = nn.Parameter(torch.rand((1,), requires_grad=True))

    def forward(self, queries, keys, values):
        # queries和attention_weights的形状为(查询数, "键-值"对数)
        queries = queries.repeat_interleave(keys.shape[1]).reshape((-1, keys.shape[1]))
        self.attention_weights = nn.functional.softmax(
            -((queries - keys) * self.w) ** 2 / 2, dim=1)
        # values的形状为(查询数, "键-值"对数)
        return torch.bmm(self.attention_weights.unsqueeze(1),
                         values.unsqueeze(-1)).reshape(-1)


## 7. 训练带参数的注意力汇聚模型


In [ ]:
# 构建训练数据
# X_tile的形状:(n_train, n_train), 每一行都包含相同的训练输入
X_tile = x_train.repeat((n_train, 1))
# Y_tile的形状:(n_train, n_train), 每一行都包含相同的训练输出
Y_tile = y_train.repeat((n_train, 1))
# keys的形状:('n_train', 'n_train'-1)
# 排除对角线元素（即自身），使用其他所有训练样本作为键
keys = X_tile[(1 - torch.eye(n_train)).type(torch.bool)].reshape((n_train, -1))
# values的形状:('n_train', 'n_train'-1)
values = Y_tile[(1 - torch.eye(n_train)).type(torch.bool)].reshape((n_train, -1))

print(f'keys shape: {keys.shape}')
print(f'values shape: {values.shape}')


In [ ]:
# 初始化模型、损失函数和优化器
net = NWKernelRegression()
loss = nn.MSELoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=0.5)

# 用于记录训练过程的损失
losses = []

# 训练循环
num_epochs = 5
for epoch in range(num_epochs):
    trainer.zero_grad()
    l = loss(net(x_train, keys, values), y_train)
    l.sum().backward()
    trainer.step()
    epoch_loss = float(l.sum())
    losses.append(epoch_loss)
    print(f'epoch {epoch + 1}, loss {epoch_loss:.6f}')


In [ ]:
# 绘制训练损失曲线
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), losses, 'o-', linewidth=2, markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('训练损失曲线')
plt.grid(True, alpha=0.3)
plt.show()

print(f'\n学习到的参数 w = {net.w.item():.4f}')


## 8. 预测与可视化


In [ ]:
# 使用训练好的模型进行预测
# keys的形状:(n_test, n_train), 每一行包含相同的训练输入（即相同的键）
keys = x_train.repeat((n_test, 1))
# values的形状:(n_test, n_train)
values = y_train.repeat((n_test, 1))
y_hat = net(x_test, keys, values).unsqueeze(1).detach()

plot_kernel_reg(y_hat)


## 9. 注意力权重可视化

与非参数的注意力汇聚模型相比，带参数的模型加入可学习的参数后，曲线在注意力权重较大的区域变得更不平滑。


In [ ]:
# 可视化带参数模型的注意力权重
show_heatmaps(net.attention_weights.unsqueeze(0).unsqueeze(0),
              xlabel='Sorted training inputs',
              ylabel='Sorted testing inputs')


## 10. 小结

- **Nadaraya-Watson核回归**是具有注意力机制的机器学习范例。
- Nadaraya-Watson核回归的**注意力汇聚**是对训练数据中输出的加权平均。从注意力的角度来看，分配给每个值的注意力权重取决于将值所对应的键和查询作为输入的函数。
- 注意力汇聚可以分为**非参数型**和**带参数型**。
- 带参数的模型通过学习参数 $w$，可以更灵活地调整注意力权重的分布。


## 11. 练习

1. 增加训练数据的样本数量，能否得到更好的非参数的Nadaraya-Watson核回归模型？
2. 在带参数的注意力汇聚的实验中学习得到的参数 $w$ 的价值是什么？为什么在可视化注意力权重时，它会使加权区域更加尖锐？
3. 如何将超参数添加到非参数的Nadaraya-Watson核回归中以实现更好地预测结果？
4. 为本节的核回归设计一个新的带参数的注意力汇聚模型。训练这个新模型并可视化其注意力权重。


In [ ]:
# 练习1: 增加训练样本数量
n_train_large = 200
x_train_large, _ = torch.sort(torch.rand(n_train_large) * 5)
y_train_large = f(x_train_large) + torch.normal(0.0, 0.5, (n_train_large,))

# 非参数注意力汇聚
X_repeat_large = x_test.repeat_interleave(n_train_large).reshape((-1, n_train_large))
attention_weights_large = nn.functional.softmax(
    -(X_repeat_large - x_train_large) ** 2 / 2, dim=1)
y_hat_large = torch.matmul(attention_weights_large, y_train_large)

plt.figure(figsize=(10, 6))
plt.plot(x_test.numpy(), y_truth.numpy(), label='Truth', linewidth=2)
plt.plot(x_test.numpy(), y_hat_large.detach().numpy(), '--', label='Pred (200 samples)', linewidth=2)
plt.scatter(x_train_large.numpy(), y_train_large.numpy(), c='red', s=10, label='Train', alpha=0.5)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.title('非参数 Nadaraya-Watson 核回归 (200个训练样本)')
plt.show()
